In [1]:
import warnings
warnings.filterwarnings('ignore')

import torch
from ase.visualize.plot import plot_atoms

from flonacomldft.internal_coordinates import Coordinates_mapping
from flonacomldft.utils.io_utils import (
    get_project_path,
    save_ase_molecules_as_traj,
    load_pickle_file
)

In [2]:
simulations_folders = {'ab-flowMC': '2-adaptive-mlp',
                       'ab-flowMC w.o. MLP': '1-adaptive',
                       'MCMC w. flow from MD': '12-mcmc-flow-md'
                      }

simulations_idx = { 'ab-flowMC': {
                        0: 32606140,
                        1: 32606143
                    },

                    'ab-flowMC w. pretrained MLP': { 
                        0: 30019620,
                        1: 30019624
                    },

                    'ab-flowMC w.o. MLP': {
                        0: 31961552,
                        1: 31961553
                    },

                    'MCMC w. flow from MD': {
                        0: 20240408004812,  
                        1: 20240408004812  
                    }

                }

def path_file(simulation_name, isomer):
    
    if 'w. pretrained MLP' in simulation_name:
        simulation_name = 'ab-flowMC' 

    idx = simulations_idx[simulation_name][isomer]

    if 'ab-flowMC' in simulation_name:

        full_path = '/'.join( (get_project_path(), simulations_folders[simulation_name],
                         'results_adaptive_is{:d}_{:d}/adaptive_sampling_is{:d}_{:d}.pkl'.format(
                             isomer, idx, isomer, idx
                         ) )
                    )

    if 'MCMC' in simulation_name:
        full_path = '/'.join( (get_project_path(), simulations_folders[simulation_name],
                         'results_multimodal_sampling_is{:d}_{:d}/is{:d}_mcmc_dic_{:d}.pkl'.format(
                             isomer, idx, isomer, idx
                         ) )
                    )        

    return full_path

In [4]:
#simulation = 'ab-flowMC'
#isomer = 1

for simulation in simulations_idx.keys():
    for isomer in [0, 1]:
        print('Path: ', path_file(simulation, isomer))
        adaptive = load_pickle_file(path_file(simulation, isomer), '')
        xs = adaptive['xs']
        if 'ab-flowMC' in simulation:
            xs = torch.cat(xs)
        xs_flatten = xs.reshape(xs.shape[0]*xs.shape[1], xs.shape[2])

        coord_mapping = Coordinates_mapping()

        trajectory = []

        for x in xs_flatten[::10]:

            molecule = coord_mapping.build_molecule_from_real_centered(x.reshape(-1, 12), isomer)[0]
            trajectory.append(molecule)

        trajectory_file_name = '{:s}_is{:d}.traj'.format(simulation, isomer).replace(' ', '_')

        save_ase_molecules_as_traj(
            trajectory,
            trajectory_file_name,
            get_project_path() + '/movies')
        
        print("Saved! {:s} is{:d}".format(simulation, isomer))


Path:  /mnt/home/amolina/ceph/adaptive-flow-mc/2-adaptive-mlp/results_adaptive_is0_32606140/adaptive_sampling_is0_32606140.pkl
Saved! ab-flowMC is0
Path:  /mnt/home/amolina/ceph/adaptive-flow-mc/2-adaptive-mlp/results_adaptive_is1_32606143/adaptive_sampling_is1_32606143.pkl
Saved! ab-flowMC is1
Path:  /mnt/home/amolina/ceph/adaptive-flow-mc/2-adaptive-mlp/results_adaptive_is0_32606140/adaptive_sampling_is0_32606140.pkl
Saved! ab-flowMC w. pretrained MLP is0
Path:  /mnt/home/amolina/ceph/adaptive-flow-mc/2-adaptive-mlp/results_adaptive_is1_32606143/adaptive_sampling_is1_32606143.pkl
Saved! ab-flowMC w. pretrained MLP is1
Path:  /mnt/home/amolina/ceph/adaptive-flow-mc/1-adaptive/results_adaptive_is0_31961552/adaptive_sampling_is0_31961552.pkl
Saved! ab-flowMC w.o. MLP is0
Path:  /mnt/home/amolina/ceph/adaptive-flow-mc/1-adaptive/results_adaptive_is1_31961553/adaptive_sampling_is1_31961553.pkl
Saved! ab-flowMC w.o. MLP is1
Path:  /mnt/home/amolina/ceph/adaptive-flow-mc/12-mcmc-flow-md/res